<a href="https://colab.research.google.com/github/julioalvarez1975/Prep-Course/blob/main/Modulo_2_clase_1_anatomia_del_prompt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clase 1 — Anatomía de un prompt

Un **prompt** es la instrucción que le das a un modelo de lenguaje. La diferencia entre una respuesta útil y una vaga casi siempre está en cómo escribiste esa instrucción — no en el modelo.

En esta clase vamos a descomponer un prompt en sus partes, entender por qué cada una importa, y comparar en tiempo real qué pasa cuando las usás bien o mal.

## Contenido

| Sección | Tema |
|---|---|
| 1 | Configuración del entorno |
| 2 | Las cinco partes de un prompt |
| 3 | Armar el wrapper LLM |
| 4 | Experimento: prompt vacío vs. prompt estructurado |
| 5 | Ajustar cada parte por separado |
| 6 | Actividad práctica |

```
DIAGRAMA ASCII — Wrapper `llamar_llm()` del notebook
====================================================

                 +----------------------------------+
                 | Usuario / notebook               |
                 | prompt, system_prompt, temp,     |
                 | max_tokens                       |
                 +----------------+-----------------+
                                  |
                                  v
                    +-----------------------------+
                    | llamar_llm(...)             |
                    |-----------------------------|
                    | if BACKEND == "gemini"      |
                    | elif BACKEND == "local"     |
                    +-------------+---------------+
                                  |
              +-------------------+-------------------+
              |                                       |
              v                                       v
+-------------------------------+      +----------------------------------+
| Rama Gemini                   |      | Rama local                       |
|-------------------------------|      |----------------------------------|
| _cliente_gemini               |      | _llm_local                       |
| .models.generate_content(...) |      | .create_chat_completion(...)     |
|                               |      |                                  |
| model = GEMINI_MODEL          |      | messages = [system, user]        |
| contents = prompt             |      | temperature                      |
| system_instruction            |      | max_tokens                       |
| temperature                   |      +----------------+-----------------+
| max_output_tokens             |                       |
+---------------+---------------+                       |
                |                                       |
                +-------------------+-------------------+
                                    |
                                    v
                     +-------------------------------+
                     | Respuesta del modelo          |
                     | .text.strip()  /  content     |
                     +---------------+---------------+
                                     |
                                     v
                     +-------------------------------+
                     | Devuelve un string final      |
                     | listo para imprimir           |
                     +-------------------------------+


Idea clave
----------
El notebook usa una sola función (`llamar_llm`) como interfaz común.
Así, el resto de las celdas no necesita cambiar aunque cambie el backend.
```

---
## 1. Configuración del entorno

Este notebook puede correr de dos formas:

| Backend | Qué necesitás | Cuándo usarlo |
|---|---|---|
| `local` | llama.cpp instalado + modelo GGUF descargado | Sin internet, máquina con ≥4 GB RAM |
| `gemini` | Cuenta Google, API key gratuita | Recomendado si es tu primera vez |

### Obtener tu API key de Gemini (solo una vez)

1. Entrá a [aistudio.google.com](https://aistudio.google.com) con tu cuenta Google.
2. Hacé clic en **Get API key** → **Create API key**.
3. Copiá la clave (empieza con `AIza...`).

### Guardar la key de forma segura

**Nunca** escribas la clave directamente en el código: si subís el notebook a GitHub, queda expuesta.
Lo correcto es guardarla en un archivo `.env` en la carpeta del proyecto:

```bash
# Ejecutá esto UNA VEZ en tu terminal, desde la carpeta del proyecto
echo 'GEMINI_API_KEY=TU_CLAVE_AQUI' >> .env
```

El archivo `.env` queda en tu máquina y **no** se sube al repositorio.
Si no querés crear el archivo, la celda siguiente te pide la clave de forma interactiva.

In [1]:
# ─── Instalación de dependencias (solo la primera vez) ────────────────────────
# Descomentá la línea que necesites:
!pip install google-genai python-dotenv          # para backend Gemini
!pip install llama-cpp-python huggingface-hub    # para backend local
print("Listo para configurar.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 11.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.0 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.23-py3-none-linux_x86_64.whl size=18716570 sha256=40008ed406517d7f81d0ceb2b67df882f5df411ad0a2a7f5c19183363235790c
  Stored in directory: /root/.cache/pip/wheels/87/c6/1e/1ecc9ef0f7ed4cadb255faa3dcb39485b26817a89b33efa2d2
Successfully built llama-cpp-python
Listo para configurar.


In [2]:
# ─── Elegí tu backend ─────────────────────────────────────────────────────────
BACKEND = "local"   # Cambiá a "local" si usás llama.cpp

In [3]:
import os
import getpass

# Modelos Gemini disponibles con tier gratuito:
GEMINI_MODEL = "gemini-2.5-flash-lite"   # rápido y gratuito
# GEMINI_MODEL = "gemma-4-26b-a4b-it"   # alternativa de Google DeepMind

# ─── Cargar API key ───────────────────────────────────────────────────────────
if BACKEND == "gemini":
    # Primero intenta leer del archivo .env
    try:
        from dotenv import load_dotenv
        load_dotenv()   # carga variables desde el archivo .env si existe
    except ImportError:
        pass  # si no está python-dotenv, sigue igual

    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")

    # Si no encontró la key en .env, la pide de forma interactiva
    if not GEMINI_API_KEY:
        GEMINI_API_KEY = getpass.getpass("Ingresá tu API key de Gemini (no se muestra): ")

print(f"Backend seleccionado: {BACKEND}")
print("API key: OK" if BACKEND == "gemini" and GEMINI_API_KEY else "(modo local)")

Backend seleccionado: local
(modo local)


---
## 2. Las cinco partes de un prompt

Un prompt bien armado puede tener hasta cinco componentes. No todos son obligatorios en todas las situaciones, pero conocerlos te permite diagnosticar rápido por qué una respuesta salió mal.

| Componente | Qué hace | Ejemplo |
|---|---|---|
| **Rol** | Le dice al modelo *quién* debe ser | `"Sos un contador público con 10 años de experiencia"` |
| **Contexto** | Información de fondo relevante para la tarea | `"El cliente tiene una PyME de importación"` |
| **Instrucción** | La tarea concreta a realizar | `"Explicá las deducciones posibles en el impuesto a las ganancias"` |
| **Formato** | Cómo debe estructurarse la respuesta | `"En 3 puntos, sin tecnicismos"` |
| **Output esperado** | Qué debe incluir o evitar la respuesta | `"Incluí un ejemplo numérico simple al final"` |

_
> 💡 El componente que más se omite (y más falta hace) es el **formato**. Sin él, el modelo elige cómo estructurar la respuesta — y no siempre elige bien para tu caso de uso.

---
## 3. Armar el wrapper LLM

Para no repetir código en cada experimento, armamos una función única `llamar_llm()` que funciona con cualquiera de los dos backends. La firma es siempre la misma: preguntás y recibís texto.

In [4]:
# ─── Inicializar el cliente según el backend elegido ─────────────────────────

if BACKEND == "gemini":
    from google import genai
    from google.genai import types

    _cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)
    print("Cliente Gemini inicializado.")

elif BACKEND == "local":
    import time
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama

    # Granite-4.0-350M — modelo instruct con chat template
    REPO_ID = "ibm-granite/granite-4.0-350m-GGUF"
    FILENAME = "granite-4.0-350m-Q4_K_M.gguf"

    print("Descargando modelo local (puede tardar la primera vez)...")
    ruta_modelo = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
    _llm_local = Llama(model_path=ruta_modelo, n_ctx=2048, n_gpu_layers=0, verbose=False)
    print("Modelo local listo.")

else:
    raise ValueError(f"Backend '{BACKEND}' no reconocido. Usá 'gemini' o 'local'.")

Descargando modelo local (puede tardar la primera vez)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


granite-4.0-350m-Q4_K_M.gguf:   0%|          | 0.00/237M [00:00<?, ?B/s]

llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Modelo local listo.


In [5]:
def llamar_llm(
    prompt,
    system_prompt="follow the instructions. answer in spanish.",
    temperature=0.7,
    max_tokens=200
):
    """Envía un prompt al modelo configurado y devuelve la respuesta como string."""

    if BACKEND == "gemini":
        respuesta = _cliente_gemini.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=temperature,
                max_output_tokens=max_tokens,
            )
        )
        return respuesta.text.strip()

    elif BACKEND == "local":
        # Granite-4.0-350M es un modelo instruct con chat template.
        # create_chat_completion() aplica el template automáticamente.
        respuesta = _llm_local.create_chat_completion(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        return respuesta["choices"][0]["message"]["content"].strip()


# Prueba rápida de conexión
print(llamar_llm("Decí 'Hola, estoy funcionando'", max_tokens=20))

Salud, me llamo asistente y estoy aquí para ayudarte con cualquier tarea o información


---
## 4. Experimento: prompt vacío vs. prompt estructurado

Vamos a hacer la misma consulta de tres formas distintas. La tarea en los tres casos es la misma: que el modelo explique qué es una red neuronal. Solo cambia cómo se la pedimos.

In [6]:
# ─── Versión 1: prompt mínimo — sin estructura ────────────────────────────────
prompt_minimo = "Explicá redes neuronales."

print("=" * 60)
print("VERSIÓN 1 — Prompt mínimo")
print("=" * 60)
print(llamar_llm(prompt_minimo))
print()

VERSIÓN 1 — Prompt mínimo
Entonces, las redes neuronales son estructuras de aprendizaje profundo que se utilizan para reconocimiento de imágenes y otros datos de alta dimensión. Estas redes aprenden a reconocer patrones y características utilizando múltiples capas de nodos conectados con intercambios de información. La capacidad de estas redes para aprender de grandes cantidades de datos les permite reconocer objetos, imágenes y otras secuencias de datos. Además, las redes neuronales también pueden aprender a realizar tareas específicas, como reconocimiento de texto, procesamiento de lenguaje natural o reconocimiento de objetos.



In [7]:
# ─── Versión 2: con rol y formato, sin contexto ───────────────────────────────
prompt_con_rol = """Sos un profesor de tecnología.
Explicá qué es una red neuronal en exactamente 3 puntos breves."""

print("=" * 60)
print("VERSIÓN 2 — Con rol y formato")
print("=" * 60)
print(llamar_llm(prompt_con_rol))
print()

VERSIÓN 2 — Con rol y formato
Un red neuronal es un sistema de aprendizaje profundo utilizado para procesar grandes cantidades de datos y aprenderá con ellos para realizar tareas como reconocimiento de imágenes o reconocimiento de voz.



In [8]:
# ─── Versión 3: prompt completo con todos los componentes ─────────────────────
prompt_completo = """Rol: Sos un profesor universitario de inteligencia artificial.
Contexto: Estás explicando a estudiantes adultos que trabajan en empresas y no tienen
  experiencia previa en programación.
Instrucción: Explicá qué es una red neuronal artificial.
Formato: 3 bullets, máximo 2 líneas cada uno, sin fórmulas matemáticas.
Output esperado: Terminá con una analogía cotidiana que resuma la idea principal."""

print("=" * 60)
print("VERSIÓN 3 — Prompt completo")
print("=" * 60)
print(llamar_llm(prompt_completo))
print()

VERSIÓN 3 — Prompt completo
Una red neuronal artificial es un modelo de aprendizaje profundo que utiliza las quefulas neuronas para aprender del mundo. Por ejemplo, podría ser como una máquina que detecta patrones en imágenes, como objetos o objetos en movimiento.



> 💡 **Para discutir:** ¿En qué versión la respuesta fue más útil para el público descrito? ¿Qué componente del prompt completo hizo más diferencia? ¿Hubo algún componente que parecía redundante?

---
## 5. Ajustar cada parte por separado

Ahora vamos a ver cómo cambia la respuesta cuando modificamos *un solo componente* a la vez. Esto entrena la intuición para saber dónde tocar cuando una respuesta no es la que esperabas.

In [9]:
# ─── Efecto del ROL ───────────────────────────────────────────────────────────
# La instrucción es idéntica, solo cambia quién se supone que responde.

instruccion_base = "Explicá en 2 oraciones por qué es importante proteger los datos personales."

roles = [
    "Sos un abogado especialista en privacidad.",
    "Sos un técnico de sistemas en una empresa mediana.",
    "Sos un periodista que escribe para el público general."
]

for rol in roles:
    print(f"Rol: {rol}")
    print("-" * 50)
    print(llamar_llm(instruccion_base, system_prompt=rol, max_tokens=120))
    print()

Rol: Sos un abogado especialista en privacidad.
--------------------------------------------------
Por qué es importante proteger los datos personales: protege a tus información personales de ser mal utilizadas, confidencial y en riesgo de daño. Asegura la confidencialidad y confianza de tus datos y mantén la privacidad de tu información personal.

Rol: Sos un técnico de sistemas en una empresa mediana.
--------------------------------------------------
1. Permite la seguridad de los datos personales: Protege los datos personales sensibles de acceso, intercambio o mal uso, asegurando que solo el titular del titular tenga acceso a ellos.
2. Asegura la confidencialidad: Protege el privacidad de los datos personales, evitando que los datos se dividen entre individuos y asegurando que solo los destinatarios autorizados puedan acceder a ellos.

Rol: Sos un periodista que escribe para el público general.
--------------------------------------------------
Por qué es importante proteger los da

In [10]:
# ─── Efecto del FORMATO ───────────────────────────────────────────────────────
# El contenido pedido es el mismo, pero el formato cambia completamente la usabilidad.

contenido = "Enumerá las ventajas de usar modelos de lenguaje en atención al cliente."

formatos = [
    "Respondé en prosa, como si fuera un párrafo de informe ejecutivo.",
    "Respondé en una lista de 4 ítems concisos, sin introducción.",
    "Respondé como una tabla con dos columnas: Ventaja | Por qué importa."
]

for fmt in formatos:
    print(f"Formato: {fmt}")
    print("-" * 50)
    print(llamar_llm(f"{contenido}\n{fmt}", max_tokens=180))
    print()

Formato: Respondé en prosa, como si fuera un párrafo de informe ejecutivo.
--------------------------------------------------
Los modelos de lenguaje en atención al cliente ofrecen numerosas ventajas que contribuyen al éxito de un negocio en línea. Primero, estos modelos pueden proporcionar una respuesta rápida y coherente a los clientes, garantizando una experiencia positiva. Además, estos modelos pueden realizar preguntas, respondiendo en tiempo real con un toque profesional. En segundo lugar, estos modelos pueden optimizar la conversación, entregando información precisa y relevante. También pueden ser entrenados para responder a diversos tipos de preguntas, haciéndolos flexible para diferentes clientes. Finalmente, los modelos de lenguaje en atención al cliente pueden mejorar el compromiso del cliente, aumentando su lealtad y satisfacción. En conclusión, estos modelos de lenguaje en atención al cliente son esenciales para una marca en línea exitosa.

Formato: Respondé en una lista d

---
## 6. Actividad práctica

Tomá la siguiente tarea y construí un prompt completo usando los cinco componentes. Después compará tu resultado con el prompt mínimo que se incluye abajo.

**Tarea:** pedirle al modelo que explique cómo funciona una tarjeta de crédito para alguien que nunca tuvo una usando el modelo local.

In [11]:
# ─── Prompt mínimo (punto de partida) ────────────────────────────────────────
prompt_minimo_actividad = "Explicá cómo funciona una tarjeta de crédito."

print("RESULTADO CON PROMPT MÍNIMO:")
print("-" * 50)
print(llamar_llm(prompt_minimo_actividad, max_tokens=150))
print()

RESULTADO CON PROMPT MÍNIMO:
--------------------------------------------------
La forma en que funciona una tarjeta de crédito es de la misma manera que otras tarjetas de crédito. Primero, el usuario solicita un crédito a través de una aplicación o portal en línea. Luego, el credit card se envía a través de este canal a una institución bancaria o de compañía de crédito. En este momento, el usuario puede recibir un correo electrónico con un número de crédito. 

Una vez que el crédito llega a la cuenta del titular del negocio, la institución de crédito procesa el pago. A continuación, la tarjeta de crédito se utiliza para pagar los préstamos y otros servicios. Si el crédito se pagó correctamente,



In [12]:
def llamar_llm(
    prompt,
    system_prompt="Responde en español, simple y conciso.",
    temperature=0.7,
    max_tokens=200 ):


    respuesta = _llm_local.create_chat_completion(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": prompt}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return respuesta["choices"][0]["message"]["content"].strip()


# Prueba rápida de conexión
print(llamar_llm("Decí 'Hola, estoy funcionando' y nada más.", max_tokens=20))

Hola, estoy funcionando con fines.


In [13]:
# TODO: Completá el prompt con los 5 componentes
# Reemplazá cada "..." con tu texto

mi_prompt = """
Rol: ...
Contexto: ...
Instrucción: ...
Formato: ...
Output esperado: ...
"""

print("RESULTADO CON TU PROMPT:")
print("-" * 50)
print(llamar_llm(mi_prompt, max_tokens=300))

RESULTADO CON TU PROMPT:
--------------------------------------------------
Rol: El rol de la persona es de administrar un sistema o tarea, como administrar un proyecto o administrar un negocio. Contexto: El contexto es un conjunto de datos o información que se utiliza para realizar un determinado propósito. Instrucción: El proceso de responder es proporcionar una respuesta clara y precisa que satisfaga la solicitud del usuario. Formato: El formato puede ser texto, gráfico, video o cualquier otro tipo de información. Output esperado: Una respuesta que resuma los puntos clave del contexto y proporciona una explicación clara de cómo se aplica la tarea.
